In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("reconciliation_table", "oh_apm_stg.tmp.ctl_reconciliation_log_Prv")
dbutils.widgets.text("error_table", "oh_apm_stg.vendor_extracts.error_load_report_log_cpc_Stg_Prv")
dbutils.widgets.text("s3_path", "s3://gia-stg-oh-ue1-data-raw/haven/inbound/VE_EDW/weekly")

In [0]:
reconciliation_table = dbutils.widgets.get("reconciliation_table")
error_table = dbutils.widgets.get("error_table")
s3_path = dbutils.widgets.get("s3_path")

In [0]:
# -*- coding: utf-8 -*-
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, LongType
import ast

# -------------------------------
# Utility function: safe get task values
# -------------------------------
def safe_get(task_key, key, default):
    try:
        return dbutils.jobs.taskValues.get(taskKey=task_key, key=key, debugValue=default)
    except Exception:
        return default

# -------------------------------
# Type coercion helpers (handle native or stringified taskValues)
# -------------------------------
def coerce_bool(x, default=False):
    if isinstance(x, bool):
        return x
    if isinstance(x, (int, float)):
        return bool(x)
    if isinstance(x, str):
        v = x.strip().lower()
        if v in ("true", "1", "yes"): return True
        if v in ("false", "0", "no"): return False
    return default

def coerce_list(x, default=None):
    if default is None: default = []
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return default
        try:
            val = ast.literal_eval(s)
            return val if isinstance(val, list) else default
        except Exception:
            return default
    return default

def coerce_dict(x, default=None):
    if default is None: default = {}
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return default
        try:
            val = ast.literal_eval(s)
            return val if isinstance(val, dict) else default
        except Exception:
            return default
    return default

def flatten_expected_names(seq):
    """
    gz_expected_info may be: list[str], list[tuple], list[dict]
    Normalize to a list[str] of file names.
    """
    out = []
    for item in seq:
        if isinstance(item, (list, tuple)) and len(item) >= 1:
            out.append(item[0])
        elif isinstance(item, dict) and "file_name" in item:
            out.append(item["file_name"])
        elif isinstance(item, str):
            out.append(item)
    return out

# -------------------------------
# Retrieve pre-validation outputs
# -------------------------------
gz_expected_info_raw = safe_get("Pre_validation", "gz_expected_info", [])
gz_file_paths_raw = safe_get("Pre_validation", "gz_file_paths", {})
missing_required_raw = safe_get("Pre_validation", "missing_required_gz_files", [])
ctl_received_ts = safe_get("Pre_validation", "ctl_received_ts", None)
start_load = safe_get("Pre_validation", "start_time", None)

# Get error table from widget
error_table = dbutils.widgets.get("error_table")

# -------------------------------
# Coerce types (robust to strings)
# -------------------------------
gz_expected_info = coerce_list(gz_expected_info_raw, default=[])
gz_file_paths = coerce_dict(gz_file_paths_raw, default={})
missing_required_gz_files = coerce_list(missing_required_raw, default=[])
expected_names = flatten_expected_names(gz_expected_info)

# -------------------------------
# Default timestamps if not set
# -------------------------------
if not start_load:
    start_load = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
end_load = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
if not ctl_received_ts:
    ctl_received_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# -------------------------------
# Prepare error records (two-phase logic)
# -------------------------------
error_records = []
phase = None

# Schema for error table
error_schema = StructType([
    StructField("File_Name", StringType(), True),
    StructField("Date_Received", StringType(), True),   # empty when missing
    StructField("Start_Load_Date", StringType(), True),
    StructField("End_Load_Date", StringType(), True),
    StructField("Row_Number", LongType(), True),
    StructField("Error_Description", StringType(), True)
])

# ---- Phase 1: CTL missing files take precedence ----
if missing_required_gz_files:
    phase = "CTL"
    for file_name in missing_required_gz_files:
        error_desc = f"MISSING REQUIRED FILE IN CTL: {file_name}"
        error_records.append((file_name, "", start_load, end_load, 0, error_desc))
else:
    # ---- Phase 2: S3 missing vs expected ----
    phase = "S3"
    keys = set(gz_file_paths.keys())
    missing_in_s3 = [name for name in expected_names if name not in keys]
    for file_name in missing_in_s3:
        error_desc = f"MISSING GZ FILE IN S3: {file_name}"
        error_records.append((file_name, "", start_load, end_load, 0, error_desc))

# -------------------------------
# Write or no-op depending on errors
# -------------------------------
if error_records:
    error_df = spark.createDataFrame(error_records, schema=error_schema)
    try:
        error_df.write.mode("append").saveAsTable(error_table)
        print(f"✅ Error report written to {error_table} (phase={phase}, count={len(error_records)})")
        display(error_df)
        dbutils.jobs.taskValues.set(key="Copy_to_APMTable", value=True)
    except Exception as e:
        raise Exception(f"❌ Failed to write error table: {e}")
else:
    print("✅ No missing files detected.")
    dbutils.jobs.taskValues.set(key="Copy_to_APMTable", value=False)

# -------------------------------
# Optional: print debug info
# -------------------------------
print(f"🔹 Phase: {phase}")
print(f"🔹 Start Load: {start_load}")
print(f"🔹 End Load: {end_load}")
print(f"🔹 Date Received: {ctl_received_ts}")

# Show raw types/values to diagnose serialization issues if any
print(f"🔹 missing_required_raw type={type(missing_required_raw)} value={missing_required_raw}")
print(f"🔹 gz_expected_info_raw type={type(gz_expected_info_raw)} value={gz_expected_info_raw}")
print(f"🔹 gz_file_paths_raw type={type(gz_file_paths_raw)} value={gz_file_paths_raw}")

# Show coerced values
print(f"🔹 Missing Required Files in CTL (coerced): {missing_required_gz_files}")
print(f"🔹 Expected GZ (names, coerced): {expected_names}")
print(f"🔹 gz_file_paths keys (count): {len(gz_file_paths)}")
if phase == "S3":
    missing_gz_files = [name for name in expected_names if name not in gz_file_paths]
    print(f"🔹 Missing GZ Files in S3: {missing_gz_files}")
print(f"🔹 Total Error Records: {len(error_records)}")
